#Presentación

El proyecto consiste en resolver una problemática presentada por un equipo de analítica de Datos de una empresa e-commerce, la cual, ha recibido la solicitud de preparar y estructurar un conjunto de datos provenientes de diversas fuentes como archivos CSV, Excel y páginas web, para su posterior análisis.

Dichos datos, presentan múltiples problemas: valores perdidos, registros duplicados, formatos inconsistentes y presencia de outliers que distorsionan los resultados.

Por tanto, el objetivo es simple; proponer una alternativa que permita transformar y limpiar estos datos, garantizando su calidad y estructuración para que puedan ser utilizados en futuros modelos predictivos y reportes de negocio.

En otras palabras, el proyecto busca desarrollar un proceso automatizado y eficiente para la obtención, limpieza, transformación, análisis y estructuración de datos utilizando Python y sus respectivas librerías: **NumPy** y **Pandas**.

Una vez desarrollado el paso a paso del proyecto, se espera contar con un dataset limpio, confiable y estructurado, listo para ser utilizado en procesos de análisis y toma de decisiones en la organización.

# La Librería numpy

En este apartado, se pretende crear un conjunto de datos ficticios utilizando NumPy, aplicando operaciones básicas para la preparación inicial.

In [ ]:
import numpy as np

#generación de datos ficticios con NumPy

np.random.seed(0) #reproductibilidad de datos ficticios
ID_clientes = np.arange(1,21) #creación de ID del 1 al 20
Ventas = np.random.randint(100,1000, size = 20) #Ventas aleatorias entre 100 y 1000
Ciudad = np.random.choice(['Santiago','Valparaíso','Talca','Concepción'], size=20) #Ciudades aleatorias

#Guardar generación de datos en formato .npy

np.save('Clientes.npy', Ventas) #Guarda el archivo

Ventas[:5], Ciudad[:5] #Visualización Parcial de los primeros 5 valores de venta y ciudades.

(array([784, 659, 729, 292, 935]),
 array(['Talca', 'Concepción', 'Concepción', 'Talca', 'Santiago'],
       dtype='<U10'))

# La librería pandas

Aquí se explora y transforma los datos generados en la lección 1 "la librería numpy", utilizando la estructura de DataFrame de Pandas.

In [ ]:
import pandas as pd

#Cargamos los datos de Numpy en un Data Frame

df = pd.DataFrame({
    'ID_clientes' : ID_clientes,
    'Ventas': Ventas,
    'Ciudad': Ciudad
}) #se construye el DataFrame con las tres columnas que integran los arrays creados con NumPy

#Exploración inicial

print(df.head()) #primeras cinco filas
print(df.describe().round(2)) #estadísticas descriptivas (por defecto) con 2 decimales
df.to_csv('Clientes.csv', index=False) #Exportar archivo en .csv sin indice automático

   ID_clientes  Ventas      Ciudad
0            1     784       Talca
1            2     659  Concepción
2            3     729  Concepción
3            4     292       Talca
4            5     935    Santiago
       ID_clientes  Ventas
count        20.00   20.00
mean         10.50  622.55
std           5.92  248.49
min           1.00  109.00
25%           5.75  447.75
50%          10.50  699.50
75%          15.25  811.00
max          20.00  935.00


# Obtención de datos desde archivos

Una vez creados y exportados como archivo CSV y NPY. Ahora se puede integrar datos de diversas fuentes y unificarlos en un solo DataFrame para su posterior limpieza.

In [ ]:
#Simulación de carga de múltiples fuentes
df_csv = pd.read_csv('Clientes.csv')

edades = np.random.randint(18, 66, size=len(df_csv))
paises = np.random.choice(['Chile', 'Perú', 'Argentina', 'México'], size=len(df_csv))
'''creación de array aleatorio de edades y países'''

df_excel = pd.DataFrame({
    'ID_clientes': df_csv['ID_clientes'],
    'Edad': edades
}) #simula la fuente de Excel con la columna Edad
df_web = pd.DataFrame({
    'ID_clientes': df_csv['ID_clientes'],
    'País': paises
}) #Simula la fuente web con la columna País

#Unificación de fuentes
'''Se utiliza pd.merge() para unir DataFrames en base a la columna común ID_clientes'''
df_unificado = pd.merge(df_csv, df_excel, on='ID_clientes', how='left')
df_unificado = pd.merge(df_unificado, df_web, on='ID_clientes', how='left')
'''El parámetro how='left' asegura que se mantengan todos los registros del CSV
y se agregue los datos de Excel y web cuando coincidan los ID.'''

#Visualización inicial del DataFrame unificado
df_unificado.head()



,ID_clientes,Ventas,Ciudad,Edad,País
0,1,784,Talca,36,Chile
1,2,659,Concepción,53,Chile
2,3,729,Concepción,42,Perú
3,4,292,Talca,47,Perú
4,5,935,Santiago,37,Argentina


# Manejo de valor perdidos y outliers

En este apartado se intenta aplicar técnicas de limpieza de datos, que resolverán una problemática común en la gestión y análitica de datos: Los valores nulos y datos atípicos.

Para ello, se ejemplificará con una modificación en la fila ventas para cada tipo de problemática, es decir, un valor nulo y un outlier.

Una vez hecho esto, se propone la solución para manejar todos aquellos valores asociados



In [ ]:
df_original = df_unificado.copy()


# Introducimos problemas simulados
df_unificado.loc[4, 'Ventas'] = np.nan #La fila 5 se convierte en nulo
df_unificado.loc[10, 'Ventas'] = 4444 #la fila 10 se convierte en un outilier extremo

print(df_unificado.head(15))



    ID_clientes  Ventas      Ciudad  Edad       País
0             1   784.0       Talca    36      Chile
1             2   659.0  Concepción    53      Chile
2             3   729.0  Concepción    42       Perú
3             4   292.0       Talca    47       Perú
4             5     NaN    Santiago    37  Argentina
5             6   863.0  Valparaíso    37      Chile
6             7   807.0  Valparaíso    32      Chile
7             8   459.0  Valparaíso    57       Perú
8             9   109.0  Valparaíso    50     México
9            10   823.0    Santiago    19      Chile
10           11  4444.0  Valparaíso    27       Perú
11           12   854.0    Santiago    50  Argentina
12           13   904.0  Concepción    49  Argentina
13           14   699.0    Santiago    28     México
14           15   170.0  Concepción    41      Chile


In [ ]:
# Guardamos una copia antes del tratamiento
df_original = df_unificado.copy()

# --- Tratamiento ---
df_unificado['Ventas'] = df_unificado['Ventas'].fillna(df_unificado['Ventas'].mean())
'''los valores nulos se reemplazan por la media de la columna Ventas'''

# Detección y tratamiento de outliers con IQR
Q1 = df_unificado['Ventas'].quantile(0.25)
Q3 = df_unificado['Ventas'].quantile(0.75)
IQR = Q3 - Q1 #Se calcula el rango intercuartilico
lim_sup = Q3 + 1.5 * IQR #se define un limite superor para detectar outliers.

df_unificado['Ventas'] = np.where(df_unificado['Ventas'] > lim_sup,
                                  df_unificado['Ventas'].median(),
                                  df_unificado['Ventas'])

# Guardar el DataFrame tratado en un nuevo archivo CSV
df_unificado.to_csv('Clientes_limpios.csv', index=False)

#Comparación simple
print("ANTES DEL TRATAMIENTO")
print(df_original['Ventas'].describe())

print("\nDESPUÉS DEL TRATAMIENTO")
print(df_unificado['Ventas'].describe())

ANTES DEL TRATAMIENTO
count      19.000000
mean      820.157895
std       909.114970
min       109.000000
25%       477.500000
50%       700.000000
75%       815.000000
max      4444.000000
Name: Ventas, dtype: float64

DESPUÉS DEL TRATAMIENTO
count     20.000000
mean     633.682895
std      236.230708
min      109.000000
25%      486.750000
50%      707.250000
75%      810.289474
max      904.000000
Name: Ventas, dtype: float64


##Data Wrangling

Además de las problemáticas anteriormente mencionadas, también se puede transformar y enriquecer los datos mediante técnicas de manipulación avanzada tales como:


*   Eliminación de registros duplicados
*   Transformar tipos de datos
*   Creación de columnas calculadas
*   Funciones personalizadas, etc.




In [ ]:
# Eliminamos duplicados para evitar filas repetidas
df_unificado = df_unificado.drop_duplicates()

# Transformamos tipos de datos estandarizando el tipo de dato para evitar error en cálculos
df_unificado['Edad'] = df_unificado['Edad'].astype('Int64')

# Creamos nuevas columnas
df_unificado['Ventas_log'] = np.log(df_unificado['Ventas'])
'''columna con el logaritmo natural de las ventas. Es Útil para análisis estadístico y normalización de datos'''

df_unificado['categoria'] = df_unificado['Ventas'].apply(lambda x: 'Alta' if x > 500 else 'Baja')
'''Se crea una columna categórica según el valor de ventas. Si la venta es mayor a 100 se clasifica como 'Alta', si no como 'Baja'''


# En este caso se aplica formato a la columna Ventas en formato $, separador de miles y sin decimales
df_unificado.head(5).style.format({"Ventas": "${:,.0f}"})



,ID_clientes,Ventas,Ciudad,Edad,País,Ventas_log,categoria
0,1,$784,Talca,36,Chile,6.664409,Alta
1,2,$659,Concepción,53,Chile,6.490724,Alta
2,3,$729,Concepción,42,Perú,6.591674,Alta
3,4,$292,Talca,47,Perú,5.676754,Baja
4,5,$820,Santiago,37,Argentina,6.709497,Alta


##Agrupamiento y Pivoteo

Finalmente, una vez tratados los datos como se quisiera generar en un dataFrame, se deve organizar y estructurar los datos para el análisis utilizando técnicas de agrupamiento y pivotado.

Para ello se considerará la última actualización que aplica la sección Data Wrangling. Se agrupará por Ciudad y Ventas.

Luego, se pretende generar una tabla dinámica que incluya Ciudad como índice, Categoría como Columna y Ventas como valores.

In [ ]:
# Agrupamiento
'''Se agrupa por ciudad y calcula la medida de ventas para que nos entrege el promedio de ventas por cada ciudad'''
agrupado = df_unificado.groupby('Ciudad')['Ventas'].mean().reset_index()

# Pivotado
'''Se crea una tabla dinámica que compare las ventas según categorías dentro de cada ciudad'''
pivotado = df_unificado.pivot_table(values='Ventas', index='Ciudad', columns='categoria', aggfunc='mean')
pivotado = pivotado.fillna(0)   # reemplaza NaN por 0

#Convertir a valores enteros la columna ventas
df_unificado['Ventas'] = df_unificado['Ventas'].astype(int)
df_unificado['Ventas_log'] = df_unificado['Ventas_log'].round(2)

# Exportación final
df_unificado.to_csv('dataset_final.csv', index=False)
df_unificado.to_excel('dataset_final.xlsx', index=False)

#Mostrar las ventas agrupadas en formato moneda
agrupado.style.format({"Ventas": "${:,.0f}"})



,Ciudad,Ventas
0,Concepción,$562
1,Santiago,$800
2,Talca,$592
3,Valparaíso,$587


In [ ]:
#Mostrar las ventas pivotadas en formato moneda
pivotado.style.format({"Alta": "${:,.0f}", "Baja": "${:,.0f}"})

categoria,Alta,Baja
Ciudad,,
Concepción,$764,$360
Santiago,$800,$0
Talca,$742,$292
Valparaíso,$739,$284


##Reflexión Final

En esta experiencia se evidenció cómo el aprendizaje práctico en el manejo de datos con Pandas y NumPy permite enfrentar situaciones reales de limpieza y transformación. Se comprendió que la simulación de errores —como valores nulos y outliers— es una estrategia útil para practicar técnicas de corrección, y que el orden en que se aplican las operaciones influye directamente en los resultados. También se identificaron problemáticas comunes, como los errores de sintaxis en nombres de columnas o la sensibilidad de la media frente a valores extremos, que obligan a ser cuidadosos y metódicos en cada paso.

Al mismo tiempo, se descubrió lo enriquecedor de complementar la parte técnica con la presentación clara de los datos: aplicar formatos de moneda, crear categorías, agrupar y pivotar información para obtener comparaciones más comprensibles. Estos procesos no solo fortalecen la capacidad de análisis, sino que también preparan los datos para ser comunicados de manera profesional y accesible. En definitiva, el aprendizaje no se limita a programar, sino que se expande hacia la construcción de narrativas con datos, resolviendo problemas y transformando información en conocimiento útil.
